# IT23183186 Model A — Varroa MobileNetV2

Notebook interface for the existing, reproducible Model A pipeline. It runs the repository scripts so the notebook and command-line workflow use the same implementation. Run cells in order. Dataset inspection and preparation are safe to run; start training and test evaluation only when ready. The current checkpoint is partial and needs retraining.

## 1. Configure paths and dependencies

Install this model environment once, if needed, using the notebook kernel: `python -m pip install -r backend/models/IT23183186_model_a_varroa_classifier/requirements.txt`. Restart the kernel after installing.

Set `DATASET_SOURCE` to the folder containing the official `train.zip`, `val.zip`, `test.zip`, and `gt.csv`. Original downloads are read-only; prepared data, reports, and checkpoints are written to separate run folders. Run cells in order. Training and final test evaluation are explicit later steps.

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

MODEL_RELATIVE_DIR = Path('backend') / 'models' / 'IT23183186_model_a_varroa_classifier'

def find_repository_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / MODEL_RELATIVE_DIR).is_dir():
            return candidate
    raise FileNotFoundError(
        f'Could not find the repository above {start}. Start Jupyter inside the project folder.'
    )

REPO_ROOT = find_repository_root()
MODEL_DIR = REPO_ROOT / MODEL_RELATIVE_DIR

# Keep this source path pointed at your unchanged official download folder.
DATASET_SOURCE = Path(r'C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_original')
PREPARED_BASE_DIR = Path(r'C:\Users\User\OneDrive\Desktop\Y4S1 NOTES\Reserach\DATAESET\VarroaDataset_prepared')
RUN_ID = datetime.now().strftime('IT23183186_ModelA_%Y%m%d_%H%M%S')
PREPARED_DIR = PREPARED_BASE_DIR / RUN_ID
REPORT_DIR = MODEL_DIR / 'reports' / RUN_ID
ARTIFACTS_DIR = MODEL_DIR / 'artifacts' / RUN_ID
for folder in (PREPARED_DIR, REPORT_DIR, ARTIFACTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable

def run(args):
    command = [str(value) for value in args]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)

print('Repository root:', REPO_ROOT)
print('Model scripts:', MODEL_DIR)
print('Dataset exists:', DATASET_SOURCE.is_dir())
print('Prepared output:', PREPARED_DIR)
print('Run artifacts:', ARTIFACTS_DIR)


## 2. Inspect the official dataset

Reports class counts, archive contents, and source metadata. Update `DATASET_SOURCE` above to your actual location first.

In [ ]:
run([PYTHON, MODEL_DIR / 'inspect_dataset.py', '--source', DATASET_SOURCE, '--verify-checksums', '--output', REPORT_DIR / 'dataset_inspection.json'])

## 3. Prepare and audit

The script preserves original files, writes prepared copies/manifests separately, and reports exclusions and duplicates.

In [ ]:
run([PYTHON, MODEL_DIR / 'prepare_dataset.py', '--source', DATASET_SOURCE, '--output', PREPARED_DIR, '--report-copy', REPORT_DIR / 'preparation_report.json'])

## 4. Train Model A

This runs the reproducible transfer-learning script: train the classification head, then fine-tune selected backbone blocks. Early stopping and best validation checkpoint saving are configured by the script. The run-specific output directory prevents overwriting earlier checkpoints. Training does not use the test split for model selection.

In [ ]:
# Run this cell when inspection and preparation have completed successfully.
run([
    PYTHON, MODEL_DIR / 'train.py',
    '--prepared-dir', PREPARED_DIR,
    '--output-dir', ARTIFACTS_DIR,
    '--seed', '2026',
    '--batch-size', '32',
    '--head-epochs', '5',
    '--finetune-epochs', '8',
    '--patience', '3',
])

## 5. Evaluate the held-out test split

Run only after training finishes successfully. This evaluates the saved checkpoint on the official test manifest, reports accuracy, per-class precision/recall/F1, confusion matrix and ROC-AUC when defined, and writes predictions. Do not use test results to tune the model.

In [ ]:
checkpoint = ARTIFACTS_DIR / 'IT23183186_ModelA_MobileNetV2.pt'
if not checkpoint.is_file():
    raise FileNotFoundError(f'Trained checkpoint not found: {checkpoint}. Complete the training cell first.')
run([
    PYTHON, MODEL_DIR / 'evaluate.py',
    '--prepared-dir', PREPARED_DIR,
    '--model-dir', ARTIFACTS_DIR,
    '--output-dir', ARTIFACTS_DIR / 'evaluation',
])